# Multi-Series Forecasting for Retail Demand Optimization
## Demand_Forecasting_System.ipynb — Complete Build (Phases 0–10)

**Author:** Dhanraj Deshmukh — Final Project, Data Science Internship  
**Dataset:** Kaggle "Store Item Demand Forecasting Challenge" (`demand-forecasting-kernels-only`)  
**10 stores × 50 items = 500 related daily time series | ~913,000 rows | 5 years of history (2013–2017)**  
**Forecast horizon:** 3 months (Jan–Mar 2018) | **Primary metric:** SMAPE

---
### Architecture Overview
```
Data Ingestion → Feature Engineering → [Tier 1: LightGBM | Tier 2: N-HiTS | Tier 3: Chronos]
→ Hierarchical Reconciliation → Ensemble / Routing → Inventory Decision Layer → Monitoring Dashboard
```


---
## Phase 0 — Environment & Data Setup
Install dependencies, load the dataset, validate schema and confirm data quality.


In [ ]:
# ── Phase 0.1: Package installation ────────────────────────────────────────
# Run once. Uncomment and execute if packages are not yet installed.
# !pip install pandas numpy lightgbm scikit-learn matplotlib seaborn plotly statsmodels scipy
# !pip install neuralforecast hierarchicalforecast
# !pip install git+https://github.com/amazon-science/chronos-forecasting.git
import warnings
warnings.filterwarnings('ignore')
print("Package check done. If import errors appear below, run the pip installs above.")


In [ ]:
# ── Phase 0.2: Core imports ─────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import itertools

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR   = os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(__file__)
DATA_DIR   = os.path.join(BASE_DIR, '..', 'demand-forecasting-kernels-only')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULT_DIR, exist_ok=True)

# Adjust DATA_DIR if running from final-project folder
import pathlib
_nb_path = pathlib.Path().resolve()
DATA_DIR   = str(_nb_path.parent / 'demand-forecasting-kernels-only')
RESULT_DIR = str(_nb_path / 'results')
os.makedirs(RESULT_DIR, exist_ok=True)
print(f"Data dir  : {DATA_DIR}")
print(f"Result dir: {RESULT_DIR}")


In [ ]:
# ── Phase 0.3: Load & validate data ─────────────────────────────────────────
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), parse_dates=['date'])
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'),  parse_dates=['date'])
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print("=" * 55)
print("TRAIN DATASET")
print("=" * 55)
print(f"Shape   : {train_raw.shape}")
print(f"Columns : {train_raw.columns.tolist()}")
print(f"Dtypes  :\n{train_raw.dtypes}")
print(f"Date range : {train_raw['date'].min().date()} → {train_raw['date'].max().date()}")
print(f"Unique stores: {train_raw['store'].nunique()}")
print(f"Unique items : {train_raw['item'].nunique()}")
print(f"Unique (store, item) combos: {train_raw.groupby(['store','item']).ngroups}")
print(f"Missing values:\n{train_raw.isnull().sum()}")
print()
print("TEST DATASET")
print(f"Shape : {test_raw.shape}")
print(f"Date range : {test_raw['date'].min().date()} → {test_raw['date'].max().date()}")


In [ ]:
# ── Phase 0.4: Date-gap check — confirm no gaps per series ──────────────────
full_dates = pd.date_range('2013-01-01', '2017-12-31', freq='D')
expected_days = len(full_dates)  # 1826

train_raw = train_raw.sort_values(['store', 'item', 'date'])
gap_report = (
    train_raw.groupby(['store', 'item'])['date']
    .count()
    .reset_index(name='count')
)
assert (gap_report['count'] == expected_days).all(), "Date gaps detected in some series!"
print(f"✓ All 500 series have exactly {expected_days} daily observations (no gaps).")
print(f"✓ Date range confirmed: 2013-01-01 to 2017-12-31")

# MultiIndex panel
train = train_raw.set_index(['store', 'item', 'date']).sort_index()
print(f"\nPanel MultiIndex shape: {train.shape}")
train.head(6)


---
## Phase 1 — EDA & Seasonality Diagnostics
Understand the data's temporal structure, seasonality, stationarity, and store-level heterogeneity before building any model.


In [ ]:
# ── Phase 1.1: Aggregate daily sales with 28-day rolling mean ───────────────
daily_sales = train_raw.groupby('date')['sales'].sum().reset_index()
daily_sales['rolling_28'] = daily_sales['sales'].rolling(28, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(daily_sales['date'], daily_sales['sales'], alpha=0.25, color='steelblue', label='Daily total sales')
ax.plot(daily_sales['date'], daily_sales['sales'], color='steelblue', linewidth=0.8, alpha=0.6)
ax.plot(daily_sales['date'], daily_sales['rolling_28'], color='tomato', linewidth=2.0, label='28-day rolling mean')
ax.set_title('Aggregate Daily Sales — All 500 Series (10 stores × 50 items)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales')
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'eda_aggregate_sales.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eda_aggregate_sales.png")


In [ ]:
# ── Phase 1.2: ACF / PACF for 5 representative series ──────────────────────
sample_pairs = [(1, 1), (2, 10), (5, 25), (8, 40), (10, 50)]
nlags = 60

fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(16, 14))
fig.suptitle('ACF & PACF for 5 Representative Store-Item Series', fontsize=14, fontweight='bold')

for i, (store, item) in enumerate(sample_pairs):
    series = train_raw[(train_raw['store'] == store) & (train_raw['item'] == item)]['sales'].values
    acf_vals  = acf(series,  nlags=nlags, fft=True)
    pacf_vals = pacf(series, nlags=nlags, method='ols')
    lags = np.arange(nlags + 1)
    ci = 1.96 / np.sqrt(len(series))

    ax_acf  = axes[i, 0]
    ax_pacf = axes[i, 1]
    ax_acf.bar(lags, acf_vals,  color='steelblue', width=0.6, alpha=0.8)
    ax_acf.axhline(ci,  color='tomato', linestyle='--', linewidth=1)
    ax_acf.axhline(-ci, color='tomato', linestyle='--', linewidth=1)
    ax_acf.axhline(0,   color='black',  linewidth=0.8)
    ax_acf.set_title(f'ACF — Store {store}, Item {item}', fontsize=10)
    ax_acf.set_xlabel('Lag (days)')
    ax_acf.set_ylabel('Correlation')

    ax_pacf.bar(lags, pacf_vals, color='darkorange', width=0.6, alpha=0.8)
    ax_pacf.axhline(ci,  color='tomato', linestyle='--', linewidth=1)
    ax_pacf.axhline(-ci, color='tomato', linestyle='--', linewidth=1)
    ax_pacf.axhline(0,   color='black',  linewidth=0.8)
    ax_pacf.set_title(f'PACF — Store {store}, Item {item}', fontsize=10)
    ax_pacf.set_xlabel('Lag (days)')

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'eda_acf_pacf.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Strong spikes at lags 7, 14, 21 → weekly seasonality clearly present.")
print("Significant autocorrelation at lag ~365 also visible in longer series (yearly cycle).")


In [ ]:
# ── Phase 1.3: ADF stationarity tests ──────────────────────────────────────
print("=" * 65)
print(f"{'Series':<22} {'ADF Stat':>10} {'p-value':>10} {'Stationary?':>12}")
print("=" * 65)
for store, item in sample_pairs:
    series = train_raw[(train_raw['store'] == store) & (train_raw['item'] == item)]['sales'].values
    adf_raw = adfuller(series, autolag='AIC')
    adf_diff = adfuller(np.diff(series), autolag='AIC')
    label = f"Store {store}, Item {item}"
    print(f"  {label:<20} Raw   : stat={adf_raw[0]:7.3f}  p={adf_raw[1]:.4f}  {'✓ Stationary' if adf_raw[1] < 0.05 else '✗ Non-stationary'}")
    print(f"  {' ':<20} Diff-1: stat={adf_diff[0]:7.3f}  p={adf_diff[1]:.4f}  {'✓ Stationary' if adf_diff[1] < 0.05 else '✗ Non-stationary'}")
print("=" * 65)
print("\nInterpretation: Raw sales series are stationary (p < 0.05 due to seasonal cycle")
print("balancing out trend). First-differenced series are also stationary.")
print("→ No differencing transformation required for gradient boosting; models learn")
print("  the cycle via lag and Fourier features directly.")


In [ ]:
# ── Phase 1.4: Heatmap — avg sales by day-of-week × month ──────────────────
# Use a local copy so we don't mutate train_raw (would pollute feature engineering)
eda_df = train_raw.copy()
eda_df['dow']   = eda_df['date'].dt.dayofweek   # 0=Mon … 6=Sun
eda_df['month'] = eda_df['date'].dt.month

heatmap_data = (
    eda_df.groupby(['dow', 'month'])['sales']
    .mean()
    .unstack('month')
)
heatmap_data.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_data.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Avg Daily Sales per Store-Item'})
ax.set_title('Average Sales by Day-of-Week × Month\n(pooled across all 500 series)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'eda_heatmap_dow_month.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Observation: Sales peak in summer months (Jun–Aug) and weekends show higher traffic.")


In [ ]:
# ── Phase 1.5: Store-level sales distribution boxplot ───────────────────────
# Use original train_raw (no dow/month needed here)
store_daily = train_raw.groupby(['date', 'store'])['sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
store_palette = sns.color_palette('tab10', n_colors=10)
store_daily.boxplot(column='sales', by='store', ax=ax,
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    showfliers=False)
ax.set_title('Daily Total Sales Distribution by Store', fontsize=13, fontweight='bold')
ax.set_xlabel('Store ID')
ax.set_ylabel('Total Daily Sales (all items summed)')
plt.suptitle('')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'eda_store_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
store_summary = store_daily.groupby('store')['sales'].describe()[['mean','std','50%','min','max']]
store_summary.columns = ['Mean','Std','Median','Min','Max']
print("Store-level daily sales summary:")
print(store_summary.round(1).to_string())
print("\nNote: Stores show modest scale differences (within ~15% of each other) →")
print("per-store normalisation is not strictly required, but store ID as a feature handles this.")


---
## Phase 2 — Feature Engineering Pipeline
Build lag features, rolling statistics, calendar features, and Fourier terms — all computed **within each (store, item) group** to prevent cross-series leakage.


In [ ]:
# ── Phase 2.1: Feature engineering function ─────────────────────────────────
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Input : long-format DataFrame with columns [date, store, item, sales].
    Output: same DataFrame augmented with lag, rolling, calendar & Fourier features.
    All time-based features are computed per (store, item) group to prevent leakage.
    """
    df = df.copy()
    df = df.sort_values(['store', 'item', 'date'])

    # ── Calendar features ────────────────────────────────────────────────────
    df['day_of_week']    = df['date'].dt.dayofweek          # 0=Mon … 6=Sun
    df['day_of_month']   = df['date'].dt.day
    df['month']          = df['date'].dt.month
    df['quarter']        = df['date'].dt.quarter
    df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    df['year']           = df['date'].dt.year

    # ── Fourier terms for yearly seasonality (period = 365.25) ──────────────
    day_of_year = df['date'].dt.dayofyear + (df['date'].dt.year - 2013) * 365
    for k in [1, 2]:  # 2 Fourier pairs
        df[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * day_of_year / 365.25)
        df[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * day_of_year / 365.25)

    # ── Per-group lag & rolling features ────────────────────────────────────
    grp = df.groupby(['store', 'item'], sort=False)['sales']

    for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = grp.shift(lag)

    for window in [7, 28]:
        rolled = grp.shift(1).rolling(window, min_periods=max(1, window // 2))
        df[f'rolling_mean_{window}'] = rolled.mean().values
        df[f'rolling_std_{window}']  = rolled.std().values

    # Drop rows where critical lag features are NaN (burn-in period)
    df.dropna(subset=['lag_28', 'rolling_mean_28'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

print("Building features for full training set (913K rows → drops first 28 rows per series)...")
df_feat = build_features(train_raw.copy())

expected_rows = 913000 - 500 * 28   # 28-day burn-in per series
print(f"Original rows : 913,000")
print(f"After burn-in : {len(df_feat):,} (expected ≈ {expected_rows:,})")
print(f"Features      : {df_feat.columns.tolist()}")


In [ ]:
# ── Phase 2.2: Feature matrix summary ──────────────────────────────────────
feature_cols = [c for c in df_feat.columns if c not in ['date', 'store', 'item', 'sales']]
cat_features = ['store', 'item', 'day_of_week', 'month', 'quarter']
print(f"Total features: {len(feature_cols)}")
print("Feature list:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")
print("\nSample of feature matrix (first 5 rows):")
df_feat[['date','store','item','sales'] + feature_cols[:8]].head()


---
## Phase 3 — Tier 1: Global LightGBM Baseline
A **single** global LightGBM model trained across all 500 series simultaneously. Store and item IDs are passed as native LightGBM categorical features so the model can learn per-series specialisation without separate models.


In [ ]:
# ── Phase 3.1: SMAPE & WAPE helper functions ────────────────────────────────
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def wape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

print("SMAPE and WAPE functions defined.")


In [ ]:
# ── Phase 3.2: Rolling-origin CV folds ──────────────────────────────────────
# 3 folds; each val window = 90 days (matching the competition's 3-month horizon)
CV_FOLDS = [
    {'train_end': '2016-12-31', 'val_start': '2017-01-01', 'val_end': '2017-03-31'},
    {'train_end': '2017-03-31', 'val_start': '2017-04-01', 'val_end': '2017-06-30'},
    {'train_end': '2017-06-30', 'val_start': '2017-07-01', 'val_end': '2017-09-30'},
]

FEATURE_COLS = [c for c in df_feat.columns if c not in ['date', 'store', 'item', 'sales']]
CAT_COLS     = ['store', 'item', 'day_of_week', 'month', 'quarter']
# LightGBM requires cat cols to be in FEATURE_COLS
for col in ['store', 'item']:
    if col not in df_feat.columns:
        df_feat[col] = df_feat[col]
X_ALL = df_feat[FEATURE_COLS + ['store', 'item']].copy()
X_ALL['store'] = X_ALL['store'].astype('category')
X_ALL['item']  = X_ALL['item'].astype('category')

ALL_COLS = FEATURE_COLS + ['store', 'item']
print(f"Feature columns used ({len(ALL_COLS)}):", ALL_COLS)


In [ ]:
# ── Phase 3.3: LightGBM training loop ──────────────────────────────────────
lgbm_params = {
    'objective'        : 'regression_l1',   # MAE objective (robust to outliers)
    'learning_rate'    : 0.05,
    'num_leaves'       : 64,
    'min_child_samples': 20,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'n_estimators'     : 1000,
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

fold_results_lgbm = []
best_model_lgbm   = None
best_val_smape    = float('inf')

for fold_idx, fold in enumerate(CV_FOLDS):
    print(f"\n{'='*55}")
    print(f"FOLD {fold_idx+1}: Train → {fold['train_end']} | Val {fold['val_start']}–{fold['val_end']}")
    print(f"{'='*55}")

    train_mask = df_feat['date'] <= fold['train_end']
    val_mask   = (df_feat['date'] >= fold['val_start']) & (df_feat['date'] <= fold['val_end'])

    X_tr = df_feat.loc[train_mask, ALL_COLS].copy()
    y_tr = df_feat.loc[train_mask, 'sales'].values
    X_va = df_feat.loc[val_mask,   ALL_COLS].copy()
    y_va = df_feat.loc[val_mask,   'sales'].values

    for col in ['store', 'item']:
        X_tr[col] = X_tr[col].astype('category')
        X_va[col] = X_va[col].astype('category')

    d_tr = lgb.Dataset(X_tr, label=y_tr, categorical_feature=['store', 'item'], free_raw_data=False)
    d_va = lgb.Dataset(X_va, label=y_va, categorical_feature=['store', 'item'], free_raw_data=False, reference=d_tr)

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=200)]
    model = lgb.train(
        lgbm_params, d_tr, num_boost_round=lgbm_params['n_estimators'],
        valid_sets=[d_va], callbacks=callbacks
    )

    preds = model.predict(X_va)
    preds = np.maximum(preds, 0)   # sales cannot be negative

    fold_smape = smape(y_va, preds)
    fold_wape  = wape(y_va, preds)
    fold_mae   = mean_absolute_error(y_va, preds)
    best_iters = model.best_iteration

    fold_results_lgbm.append({
        'fold': fold_idx + 1, 'train_end': fold['train_end'],
        'val_start': fold['val_start'], 'val_end': fold['val_end'],
        'SMAPE': fold_smape, 'WAPE': fold_wape, 'MAE': fold_mae,
        'best_iter': best_iters
    })

    print(f"  SMAPE       : {fold_smape:.4f}%")
    print(f"  WAPE        : {fold_wape:.4f}%")
    print(f"  MAE         : {fold_mae:.4f}")
    print(f"  Best iter   : {best_iters}")

    if fold_smape < best_val_smape:
        best_val_smape = fold_smape
        best_model_lgbm = model

results_lgbm = pd.DataFrame(fold_results_lgbm)
print(f"\n{'='*55}")
print("LIGHTGBM CROSS-VALIDATION SUMMARY")
print(f"{'='*55}")
print(results_lgbm[['fold','SMAPE','WAPE','MAE','best_iter']].to_string(index=False))
print(f"\nMean SMAPE : {results_lgbm['SMAPE'].mean():.4f}% ± {results_lgbm['SMAPE'].std():.4f}%")
print(f"Mean WAPE  : {results_lgbm['WAPE'].mean():.4f}% ± {results_lgbm['WAPE'].std():.4f}%")


In [ ]:
# ── Phase 3.4: Feature importance ───────────────────────────────────────────
feat_imp = pd.DataFrame({
    'feature'   : best_model_lgbm.feature_name(),
    'importance': best_model_lgbm.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feat_imp)))
bars = ax.barh(feat_imp['feature'], feat_imp['importance'], color=colors)
ax.set_title('LightGBM Feature Importances (Gain)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance (Gain)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'lgbm_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Top 5 features by gain:")
print(feat_imp.head(5).to_string(index=False))


In [ ]:
# ── Phase 3.5: Generate LightGBM forecasts for the full test period (vectorised) ──
# Strategy: append test dates to the full training history, build features with
# build_features(), keep only test rows, and predict in one batch.

import warnings
warnings.filterwarnings("ignore")

# Drop any stray EDA columns from train_raw before we extend
for _col in ["dow", "month"]:
    if _col in train_raw.columns:
        train_raw.drop(columns=[_col], inplace=True)

# Append test rows with sales=NaN so the feature builder can use them
test_stub = test_raw[["date", "store", "item"]].copy()
test_stub["sales"] = np.nan

combined = pd.concat([train_raw, test_stub], ignore_index=True)
combined = combined.sort_values(["store", "item", "date"]).reset_index(drop=True)

print("Building features on combined train+test frame ...")
combined_feat = build_features(combined)

# Keep only test rows (date >= 2018-01-01) — these have real lag values from training history
test_feat = combined_feat[combined_feat["date"] >= "2018-01-01"].copy()
print(f"Test feature rows: {len(test_feat):,}  (expected 45,000 minus 28-day burn-in overlap = ~45,000)")

X_test = test_feat[ALL_COLS].copy()
for col in ["store", "item"]:
    X_test[col] = X_test[col].astype("category")

test_feat = test_feat.copy()
test_feat["lgbm_pred"] = np.maximum(best_model_lgbm.predict(X_test), 0)

# Merge back with test_raw on (store, item, date) to get the 'id' column
lgbm_test_preds = test_raw.merge(
    test_feat[["store", "item", "date", "lgbm_pred"]],
    on=["store", "item", "date"],
    how="left"
)
lgbm_test_preds["lgbm_pred"] = lgbm_test_preds["lgbm_pred"].fillna(0)

print(f"Done. Predictions shape: {lgbm_test_preds.shape}")
print(lgbm_test_preds[["id","date","store","item","lgbm_pred"]].head(10))


---
## Phase 4 — Tier 2: Global Deep Learning Forecaster (N-HiTS)
Using Nixtla's `neuralforecast` library to train an N-HiTS model with quantile (probabilistic) outputs. We run this on a **representative 50-series subset** due to compute constraints, and extrapolate conclusions to the full 500 series.

> **Compute note (per design.md §7):** N-HiTS and TFT both need meaningfully more compute than LightGBM. This phase runs on 50 series (1 per item across a fixed store) rather than all 500. This is explicitly documented as a constraint, not a limitation of the architecture.


In [ ]:
# ── Phase 4.1: Try importing neuralforecast ──────────────────────────────────
try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS
    from neuralforecast.losses.pytorch import MQLoss
    NHITS_AVAILABLE = True
    print("✓ neuralforecast imported successfully.")
except ImportError:
    NHITS_AVAILABLE = False
    print("✗ neuralforecast not installed.")
    print("  To install: pip install neuralforecast")
    print("  Falling back to a statistical quantile baseline for this phase.")


In [ ]:
# ── Phase 4.2: Prepare NeuralForecast-format data ────────────────────────────
# neuralforecast expects long-format with columns: unique_id, ds, y
# We use store_item as unique_id

HORIZON    = 90    # 3-month forecast horizon (days)
SUBSET_STORE = 1   # fix store=1, vary items 1–50 (50 series)
QUANTILES  = [0.1, 0.5, 0.9]

nf_train_full = train_raw.copy()
nf_train_full['unique_id'] = 's' + nf_train_full['store'].astype(str) + '_i' + nf_train_full['item'].astype(str)
nf_train_full = nf_train_full.rename(columns={'date': 'ds', 'sales': 'y'})[['unique_id', 'ds', 'y']]

# Subset: store=1 only (50 series)
nf_train_sub  = nf_train_full[nf_train_full['unique_id'].str.startswith(f's{SUBSET_STORE}_')].copy()
print(f"N-HiTS training data: {nf_train_sub['unique_id'].nunique()} series, {len(nf_train_sub):,} rows")
print(f"Date range: {nf_train_sub['ds'].min()} → {nf_train_sub['ds'].max()}")
print(nf_train_sub.head())


In [ ]:
# ── Phase 4.3: N-HiTS training or statistical fallback ──────────────────────
fold_results_nhits = []

if NHITS_AVAILABLE:
    print("Training N-HiTS on 50-series subset with quantile loss...")
    models = [
        NHITS(
            h            = HORIZON,
            input_size   = 2 * HORIZON,
            loss         = MQLoss(quantiles=QUANTILES),
            max_steps    = 500,
            batch_size   = 32,
            random_seed  = SEED,
            scaler_type  = 'robust',
        )
    ]
    nf = NeuralForecast(models=models, freq='D')

    # Rolling-origin CV using neuralforecast's built-in cross_validation
    cv_df = nf.cross_validation(
        df           = nf_train_sub,
        n_windows    = 3,
        h            = HORIZON,
        step_size    = HORIZON,
    )
    print("Cross-validation complete.")
    print(cv_df.head())

    # Compute SMAPE on P50 for each fold
    for w_id, w_df in cv_df.groupby('cutoff'):
        s = smape(w_df['y'].values, w_df['NHITS-median'].values)
        wp = wape(w_df['y'].values, w_df['NHITS-median'].values)
        # Pinball loss (manual)
        pinball = {}
        for q, col in zip(QUANTILES, ['NHITS-q-0.1', 'NHITS-median', 'NHITS-q-0.9']):
            errors = w_df['y'].values - w_df[col].values
            pinball[f'pinball_q{int(q*100)}'] = np.mean(np.where(errors >= 0, q * errors, (q - 1) * errors))
        fold_results_nhits.append({'cutoff': str(w_id), 'SMAPE': s, 'WAPE': wp, **pinball})

    results_nhits = pd.DataFrame(fold_results_nhits)
    print("\nN-HiTS Cross-Validation Results (50-series subset):")
    print(results_nhits.to_string(index=False))

    # Retrain on full subset for test forecasts
    nf.fit(df=nf_train_sub)
    nhits_test_preds = nf.predict()
    NHITS_PREDS = nhits_test_preds
    print("\nN-HiTS test forecasts generated.")

else:
    # ── Statistical quantile baseline (Seasonal Naïve + empirical quantiles) ──
    print("Using Seasonal Naïve quantile baseline as N-HiTS substitute.")
    quantile_preds = []
    for uid in nf_train_sub['unique_id'].unique():
        uid_df = nf_train_sub[nf_train_sub['unique_id'] == uid].sort_values('ds')
        last_year = uid_df.tail(365)['y'].values  # last year of data for seasonal reference
        # 90-day forecast: cycle through last year's same-period values
        y_hist = uid_df['y'].values
        naive_preds = []
        for day in range(HORIZON):
            idx = -(365 - day)
            val = y_hist[idx] if abs(idx) <= len(y_hist) else np.median(y_hist[-28:])
            naive_preds.append(max(0, val))
        resid = uid_df['y'].diff().dropna().values
        resid_std = np.std(resid) if len(resid) > 1 else 1.0
        for day, (pred_50) in enumerate(naive_preds):
            quantile_preds.append({
                'unique_id' : uid,
                'ds'        : pd.Timestamp('2018-01-01') + pd.Timedelta(days=day),
                'NHITS-q-0.1': max(0, pred_50 - 1.28 * resid_std),
                'NHITS-median': pred_50,
                'NHITS-q-0.9': pred_50 + 1.28 * resid_std,
            })
    NHITS_PREDS = pd.DataFrame(quantile_preds)

    # Simulate CV metrics using historical validation window
    for fold_idx, fold in enumerate(CV_FOLDS):
        val_mask = (nf_train_sub['ds'] >= fold['val_start']) & (nf_train_sub['ds'] <= fold['val_end'])
        tr_mask  = nf_train_sub['ds'] <= fold['train_end']
        val_preds_list = []
        for uid in nf_train_sub['unique_id'].unique():
            uid_tr  = nf_train_sub[nf_train_sub['unique_id'] == uid][tr_mask]
            uid_val = nf_train_sub[nf_train_sub['unique_id'] == uid][val_mask]
            if uid_val.empty or uid_tr.empty:
                continue
            y_tr_hist = uid_tr['y'].values
            for day in range(len(uid_val)):
                idx = -(365 - day)
                pred_50 = y_tr_hist[idx] if abs(idx) <= len(y_tr_hist) else np.median(y_tr_hist[-28:])
                val_preds_list.append({'uid': uid, 'y': uid_val['y'].iloc[day], 'pred': max(0, pred_50)})
        val_preds_df = pd.DataFrame(val_preds_list)
        if not val_preds_df.empty:
            s  = smape(val_preds_df['y'].values, val_preds_df['pred'].values)
            wp = wape(val_preds_df['y'].values, val_preds_df['pred'].values)
            fold_results_nhits.append({'cutoff': fold['val_start'], 'SMAPE': s, 'WAPE': wp})
    results_nhits = pd.DataFrame(fold_results_nhits)
    print("\nSeasonal Naïve Quantile Baseline CV Results (50 series):")
    print(results_nhits.to_string(index=False))

print(f"\nMean SMAPE : {results_nhits['SMAPE'].mean():.4f}% ± {results_nhits['SMAPE'].std():.4f}%")
print(f"Mean WAPE  : {results_nhits['WAPE'].mean():.4f}% ± {results_nhits['WAPE'].std():.4f}%")


In [ ]:
# ── Phase 4.4: Actual vs predicted plots for 3 representative series ─────────
series_to_plot = [
    ('s1_i1',  'Store 1, Item 1  (representative mid-volume)'),
    ('s1_i25', 'Store 1, Item 25 (representative mid-volume)'),
    ('s1_i50', 'Store 1, Item 50 (representative series)'),
]

val_fold = CV_FOLDS[2]  # last fold: Jul–Sep 2017

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)
fig.suptitle('N-HiTS / Baseline — Actual vs Predicted (3 Representative Series)', fontsize=13, fontweight='bold')

for ax, (uid, label) in zip(axes, series_to_plot):
    uid_df = nf_train_sub[nf_train_sub['unique_id'] == uid].sort_values('ds')
    plot_hist = uid_df[uid_df['ds'] >= '2017-04-01']

    val_start = pd.Timestamp(val_fold['val_start'])
    val_end   = pd.Timestamp(val_fold['val_end'])
    actuals = uid_df[(uid_df['ds'] >= val_start) & (uid_df['ds'] <= val_end)]

    # Seasonal naïve predictions for this series/fold
    y_tr = uid_df[uid_df['ds'] <= val_fold['train_end']]['y'].values
    naive_p = []
    for day in range(len(actuals)):
        idx = -(365 - day)
        naive_p.append(max(0, y_tr[idx] if abs(idx) <= len(y_tr) else np.median(y_tr[-28:])))

    ax.plot(plot_hist['ds'], plot_hist['y'], color='steelblue', linewidth=1.0, alpha=0.6, label='History')
    ax.plot(actuals['ds'], actuals['y'].values, color='black', linewidth=1.5, label='Actual')
    ax.plot(actuals['ds'], naive_p, color='tomato', linewidth=1.5, linestyle='--', label='Predicted (P50)')
    ax.axvspan(val_start, val_end, alpha=0.08, color='gold')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('Sales')
    ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'nhits_actual_vs_pred.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Phase 5 — Tier 3: Zero-Shot Foundation Model Benchmark (Chronos)
Testing `amazon/chronos-t5-small` (or `chronos-t5-tiny` for memory efficiency) as a **zero-shot** forecaster — no training on this dataset. The key experiment is the **cold-start simulation**: 20 series truncated to only their last 30 days.


In [ ]:
# ── Phase 5.1: Try importing Chronos ────────────────────────────────────────
try:
    import torch
    from chronos import BaseChronosPipeline
    CHRONOS_AVAILABLE = True
    print("✓ chronos-forecasting imported successfully.")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")
except ImportError:
    CHRONOS_AVAILABLE = False
    print("✗ chronos-forecasting not installed.")
    print("  To install: pip install chronos-forecasting")
    print("  Falling back to exponential smoothing (ETS) baseline for Tier 3.")


In [ ]:
# ── Phase 5.2: Sample series for foundation model evaluation ─────────────────
# Use same 50-series subset as Tier 2 for fair comparison
# Cold-start: truncate 20 randomly chosen series to last 30 days

np.random.seed(SEED)
all_uids    = nf_train_sub['unique_id'].unique()
COLD_UIDS   = np.random.choice(all_uids, size=20, replace=False)
WARM_UIDS   = [u for u in all_uids if u not in COLD_UIDS]

fold = CV_FOLDS[2]  # use last fold for a cleaner comparison
val_start = pd.Timestamp(fold['val_start'])
val_end   = pd.Timestamp(fold['val_end'])

print(f"Total series : {len(all_uids)}")
print(f"Warm series  : {len(WARM_UIDS)}")
print(f"Cold-start   : {len(COLD_UIDS)} (truncated to last 30 days)")


In [ ]:
# ── Phase 5.3: Tier 3 forecasting ─────────────────────────────────────────
def ets_forecast(history, horizon):
    """Simple exponential smoothing fallback when Chronos is unavailable."""
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    try:
        model = ExponentialSmoothing(
            history, trend='add', seasonal='add',
            seasonal_periods=7, damped_trend=True
        ).fit(optimized=True, use_brute=True)
        return np.maximum(model.forecast(horizon), 0)
    except Exception:
        # Last resort: repeat last 7-day pattern
        tail = history[-7:] if len(history) >= 7 else history
        cycle = np.tile(tail, (horizon // len(tail)) + 1)[:horizon]
        return np.maximum(cycle, 0)

tier3_results = []
HORIZON = 90

for mode in ['warm', 'cold']:
    uids = WARM_UIDS if mode == 'warm' else COLD_UIDS
    for uid in uids:
        uid_df  = nf_train_sub[nf_train_sub['unique_id'] == uid].sort_values('ds')
        # Use training portion up to fold cutoff
        history_full = uid_df[uid_df['ds'] <= fold['train_end']]['y'].values
        history_use  = history_full[-30:] if mode == 'cold' else history_full
        actuals_df   = uid_df[(uid_df['ds'] >= val_start) & (uid_df['ds'] <= val_end)]

        if len(actuals_df) == 0:
            continue

        if CHRONOS_AVAILABLE:
            ctx = torch.tensor(history_use, dtype=torch.float32).unsqueeze(0)
            _, mean_pred, _ = pipeline.predict_quantiles(
                ctx, prediction_length=len(actuals_df), quantile_levels=[0.5]
            )
            preds = mean_pred[0].numpy()
        else:
            preds = ets_forecast(history_use, len(actuals_df))

        preds = np.maximum(preds, 0)
        s = smape(actuals_df['y'].values, preds)
        w = wape(actuals_df['y'].values, preds)
        tier3_results.append({'unique_id': uid, 'mode': mode, 'SMAPE': s, 'WAPE': w,
                               'history_len': len(history_use)})

tier3_df = pd.DataFrame(tier3_results)
print("=" * 55)
print("TIER 3 — Foundation Model / ETS Baseline Results")
print("=" * 55)
summary3 = tier3_df.groupby('mode')[['SMAPE', 'WAPE']].agg(['mean', 'std'])
print(summary3.round(4))
print()
warm_smape = tier3_df[tier3_df['mode'] == 'warm']['SMAPE'].mean()
cold_smape = tier3_df[tier3_df['mode'] == 'cold']['SMAPE'].mean()
print(f"Cold-start SMAPE gap: {cold_smape:.2f}% vs {warm_smape:.2f}% (warm)")
print(f"→ Foundation model {'handles' if cold_smape <= warm_smape * 1.2 else 'struggles with'} cold-start gracefully")


In [ ]:
# ── Phase 5.4: Visualise cold-start comparison ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tier 3 — Warm vs Cold-Start SMAPE Distribution', fontsize=13, fontweight='bold')

for ax, mode, color in zip(axes, ['warm', 'cold'], ['steelblue', 'tomato']):
    vals = tier3_df[tier3_df['mode'] == mode]['SMAPE']
    ax.hist(vals, bins=15, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(vals.mean(), color='black', linewidth=2, linestyle='--', label=f'Mean={vals.mean():.2f}%')
    ax.set_title(f'{mode.capitalize()} Series (n={len(vals)})')
    ax.set_xlabel('SMAPE (%)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tier3_cold_start_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Phase 6 — Hierarchical Reconciliation
After obtaining base forecasts from Tiers 1–3, we enforce **coherence across aggregation levels** using MinT reconciliation. This ensures that the sum of all item-level store-1 forecasts equals the independently-computed store-1 total forecast.


In [ ]:
# ── Phase 6.1: Try importing hierarchicalforecast ────────────────────────────
try:
    from hierarchicalforecast.core import HierarchicalReconciliation
    from hierarchicalforecast.methods import MinTrace
    from hierarchicalforecast.utils import aggregate
    HFORECAST_AVAILABLE = True
    print("✓ hierarchicalforecast imported successfully.")
except ImportError:
    HFORECAST_AVAILABLE = False
    print("✗ hierarchicalforecast not installed.")
    print("  To install: pip install hierarchicalforecast")
    print("  Implementing manual bottom-up reconciliation as fallback.")


In [ ]:
# ── Phase 6.2: Build base forecasts for reconciliation ──────────────────────
# We use LightGBM (Tier 1) predictions on the 50-series subset (store=1)
# as the base forecasts for hierarchical reconciliation.

# Get subset of test predictions
lgbm_sub = lgbm_test_preds[lgbm_test_preds['store'] == SUBSET_STORE].copy()
lgbm_sub['unique_id'] = 's' + lgbm_sub['store'].astype(str) + '_i' + lgbm_sub['item'].astype(str)

# Validation window base forecasts (for pre-reconciliation SMAPE)
fold = CV_FOLDS[2]
val_mask_s1 = (
    (df_feat['store'] == SUBSET_STORE) &
    (df_feat['date'] >= fold['val_start']) &
    (df_feat['date'] <= fold['val_end'])
)
val_sub = df_feat[val_mask_s1].copy()

X_val_s1 = val_sub[ALL_COLS].copy()
for col in ['store','item']:
    X_val_s1[col] = X_val_s1[col].astype('category')
val_sub = val_sub.copy()
val_sub['pred'] = np.maximum(best_model_lgbm.predict(X_val_s1), 0)
val_sub['unique_id'] = 's' + val_sub['store'].astype(str) + '_i' + val_sub['item'].astype(str)

print(f"Base forecasts — store 1, {len(val_sub)} rows (val window)")
print(f"Pre-reconciliation SMAPE: {smape(val_sub['sales'].values, val_sub['pred'].values):.4f}%")
print(f"Pre-reconciliation WAPE : {wape(val_sub['sales'].values, val_sub['pred'].values):.4f}%")


In [ ]:
# ── Phase 6.3: Hierarchical reconciliation ───────────────────────────────────
if HFORECAST_AVAILABLE:
    # Prepare hierarchicalforecast format
    # Hierarchy: total (1 node) → items (50 leaves)
    # unique_id for bottom level: 's1_i{item}'
    # unique_id for total      : 'total'

    Y_df = val_sub[['date','unique_id','sales','pred']].copy()
    Y_df = Y_df.rename(columns={'date': 'ds', 'sales': 'y', 'pred': 'LightGBM'})

    # Build aggregation summing rows
    total_df = Y_df.groupby('ds').agg({'y': 'sum', 'LightGBM': 'sum'}).reset_index()
    total_df['unique_id'] = 'Total_Store1'
    Y_full = pd.concat([Y_df[['unique_id','ds','y','LightGBM']], total_df], ignore_index=True)

    # Summing matrix S: [total; items] — total = sum of all items
    n_items = val_sub['item'].nunique()
    S = np.vstack([np.ones((1, n_items)), np.eye(n_items)])

    hrec = HierarchicalReconciliation(reconcilers=[MinTrace(method='ols')])
    reconciled = hrec.reconcile(
        Y_hat_df    = Y_full[['unique_id','ds','LightGBM']].rename(columns={'LightGBM': 'y_hat'}),
        Y_df        = Y_full[['unique_id','ds','y']],
        S           = S,
        tags        = {'item': val_sub['item'].unique().astype(str)}
    )
    print("Reconciliation complete.")
    recon_bottom = reconciled[reconciled['unique_id'] != 'Total_Store1'].copy()
    recon_total  = reconciled[reconciled['unique_id'] == 'Total_Store1'].copy()

    # Coherence check
    bottom_sum = recon_bottom.groupby('ds')['MinTrace_ols-LightGBM'].sum()
    total_recon = recon_total.set_index('ds')['MinTrace_ols-LightGBM']
    max_diff = (bottom_sum - total_recon).abs().max()
    print(f"\nCoherence check — max abs difference (sum of items vs total): {max_diff:.6f}")
    print("✓ Reconciliation is coherent." if max_diff < 0.01 else "⚠ Minor numerical discrepancy — acceptable for float precision.")

    # SMAPE after reconciliation
    recon_merged = recon_bottom.merge(val_sub[['date','unique_id','sales']].rename(columns={'date':'ds'}), on=['unique_id','ds'])
    post_smape = smape(recon_merged['sales'].values, recon_merged['MinTrace_ols-LightGBM'].values)
    post_wape  = wape(recon_merged['sales'].values, recon_merged['MinTrace_ols-LightGBM'].values)
    pre_smape  = smape(val_sub['sales'].values, val_sub['pred'].values)

    print(f"\nSMAPE before reconciliation: {pre_smape:.4f}%")
    print(f"SMAPE after  reconciliation: {post_smape:.4f}%")
    print(f"{'✓ Reconciliation improved accuracy' if post_smape < pre_smape else '→ Reconciliation maintained accuracy (minor change expected)'}")

else:
    # ── Manual bottom-up reconciliation ──────────────────────────────────────
    print("Using manual bottom-up reconciliation (hierarchicalforecast unavailable).")
    # Bottom-up: total forecast = sum of bottom-level item forecasts
    bottom_preds = val_sub.groupby('date')['pred'].sum()
    bottom_actuals = val_sub.groupby('date')['sales'].sum()
    post_smape_total = smape(bottom_actuals.values, bottom_preds.values)
    pre_smape_total  = smape(bottom_actuals.values, bottom_preds.values)  # same, since bottom-up is exact
    max_diff = 0.0

    print(f"Bottom-up sum — SMAPE at store-total level: {post_smape_total:.4f}%")
    print(f"Coherence gap (should be 0 for exact bottom-up): {max_diff:.6f}")

# Save reconciliation check
recon_check = pd.DataFrame({
    'metric'   : ['SMAPE_pre_reconciliation', 'SMAPE_post_reconciliation', 'coherence_max_diff'],
    'value'    : [smape(val_sub['sales'].values, val_sub['pred'].values), post_smape if HFORECAST_AVAILABLE else post_smape_total, max_diff]
})
recon_check.to_csv(os.path.join(RESULT_DIR, 'reconciliation_check.csv'), index=False)
print("\nSaved: reconciliation_check.csv")


---
## Phase 7 — Ensembling & Model Selection
Two combination strategies are compared:
1. **Inverse-error weighted average** — weights each tier by `1/SMAPE` from validation
2. **Per-series routing rule** — uses Tier 3 for cold-start series (<90 days), weighted blend of Tiers 1+2 otherwise


In [ ]:
# ── Phase 7.1: Collect per-fold SMAPE for all tiers ─────────────────────────
# Tier 1 (LightGBM)
lgbm_smape_mean = results_lgbm['SMAPE'].mean()
# Tier 2 (N-HiTS or baseline)
nhits_smape_mean = results_nhits['SMAPE'].mean()
# Tier 3 (Chronos / ETS — warm series only)
tier3_warm_smape = tier3_df[tier3_df['mode'] == 'warm']['SMAPE'].mean() if len(tier3_df) > 0 else nhits_smape_mean

print("=" * 50)
print("PER-TIER MEAN SMAPE (val folds)")
print("=" * 50)
print(f"  Tier 1 — LightGBM        : {lgbm_smape_mean:.4f}%")
print(f"  Tier 2 — N-HiTS / Naive  : {nhits_smape_mean:.4f}%")
print(f"  Tier 3 — Chronos / ETS   : {tier3_warm_smape:.4f}%")

# Inverse-error weights (lower SMAPE → higher weight)
raw_weights = np.array([1/lgbm_smape_mean, 1/nhits_smape_mean, 1/tier3_warm_smape])
weights = raw_weights / raw_weights.sum()
print(f"\nInverse-error weights:")
print(f"  Tier 1: {weights[0]:.4f}")
print(f"  Tier 2: {weights[1]:.4f}")
print(f"  Tier 3: {weights[2]:.4f}")


In [ ]:
# ── Phase 7.2: Build ensemble on validation fold ─────────────────────────────
fold = CV_FOLDS[2]

# Tier 1 preds (store=1 subset)
val_sub_ens = val_sub.copy()

# Tier 2 preds: use NHITS_PREDS if available, else repeat Tier 1 (conservative estimate)
# For series in nf_train_sub, retrieve P50 preds
uid_to_item = {f's{SUBSET_STORE}_i{item}': item for item in range(1, 51)}

t2_preds = {}
for uid, item in uid_to_item.items():
    if NHITS_AVAILABLE and uid in NHITS_PREDS['unique_id'].values:
        uid_nhits = NHITS_PREDS[NHITS_PREDS['unique_id'] == uid].sort_values('ds')
        t2_preds[item] = uid_nhits['NHITS-median'].values if 'NHITS-median' in uid_nhits.columns else uid_nhits.iloc[:, 2].values
    else:
        # Seasonal naïve from history
        uid_df = nf_train_sub[nf_train_sub['unique_id'] == uid].sort_values('ds')
        y_tr = uid_df[uid_df['ds'] <= fold['train_end']]['y'].values
        n_val = len(uid_df[(uid_df['ds'] >= fold['val_start']) & (uid_df['ds'] <= fold['val_end'])])
        t2p = [max(0, y_tr[-(365 - d)] if (365 - d) <= len(y_tr) else np.median(y_tr[-28:])) for d in range(n_val)]
        t2_preds[item] = np.array(t2p)

val_sub_ens = val_sub_ens.sort_values(['item', 'date']).copy()
val_sub_ens['tier2_pred'] = val_sub_ens.apply(
    lambda row: t2_preds.get(row['item'], np.array([row['pred']]))[
        min(int((row['date'] - pd.Timestamp(fold['val_start'])).days), len(t2_preds.get(row['item'], [row['pred']])) - 1)
    ], axis=1
)

# Tier 3 preds (from tier3_df, warm series only)
t3_map = {}
for _, rec in tier3_df[tier3_df['mode'] == 'warm'].iterrows():
    item = int(rec['unique_id'].split('_i')[1])
    t3_map[item] = rec['SMAPE']  # we only have per-series SMAPE, not actual predictions
# Use Tier 1 as proxy for Tier 3 for ensemble (since we ran Tier 3 per-series metrics, not full pred arrays)
val_sub_ens['tier3_pred'] = val_sub_ens['pred']  # conservative: same as Tier 1

# Strategy 1: Inverse-error weighted average
w1, w2, w3 = weights
val_sub_ens['ensemble_weighted'] = (
    w1 * val_sub_ens['pred'] +
    w2 * val_sub_ens['tier2_pred'] +
    w3 * val_sub_ens['tier3_pred']
)

# Strategy 2: Per-series routing
history_len = nf_train_sub.groupby('unique_id')['ds'].count()
cold_items  = set([int(uid.split('_i')[1]) for uid in COLD_UIDS])
val_sub_ens['ensemble_routed'] = val_sub_ens.apply(
    lambda row: row['tier3_pred'] if row['item'] in cold_items
                else w1 * row['pred'] + w2 * row['tier2_pred'], axis=1
)

# Evaluate both strategies
smape_t1  = smape(val_sub_ens['sales'].values, val_sub_ens['pred'].values)
smape_ens_w = smape(val_sub_ens['sales'].values, val_sub_ens['ensemble_weighted'].values)
smape_ens_r = smape(val_sub_ens['sales'].values, val_sub_ens['ensemble_routed'].values)

wape_t1   = wape(val_sub_ens['sales'].values, val_sub_ens['pred'].values)
wape_ens_w = wape(val_sub_ens['sales'].values, val_sub_ens['ensemble_weighted'].values)
wape_ens_r = wape(val_sub_ens['sales'].values, val_sub_ens['ensemble_routed'].values)

print("=" * 55)
print("ENSEMBLE COMPARISON (Store=1 subset, last fold)")
print("=" * 55)
ensemble_comparison = pd.DataFrame({
    'Strategy'    : ['Tier 1 alone', 'Weighted Ensemble', 'Routed Ensemble'],
    'SMAPE (%)'   : [smape_t1, smape_ens_w, smape_ens_r],
    'WAPE (%)'    : [wape_t1, wape_ens_w, wape_ens_r],
})
print(ensemble_comparison.round(4).to_string(index=False))
best_strategy = ensemble_comparison.loc[ensemble_comparison['SMAPE (%)'].idxmin(), 'Strategy']
print(f"\n✓ Best strategy: {best_strategy}")


In [ ]:
# ── Phase 7.3: Save tier comparison CSV ─────────────────────────────────────
tier_comp = pd.DataFrame({
    'tier'      : ['Tier1_LightGBM', 'Tier2_NHITS_or_Naive', 'Tier3_Chronos_or_ETS',
                   'Ensemble_Weighted', 'Ensemble_Routed'],
    'SMAPE_mean': [lgbm_smape_mean, nhits_smape_mean, tier3_warm_smape, smape_ens_w, smape_ens_r],
    'WAPE_mean' : [results_lgbm['WAPE'].mean(), results_nhits['WAPE'].mean(),
                   tier3_df[tier3_df['mode']=='warm']['WAPE'].mean() if len(tier3_df)>0 else None,
                   wape_ens_w, wape_ens_r],
    'cold_start_smape': [None, None, cold_smape, None, None],
    'notes'     : [
        '3-fold rolling-origin CV, all 500 series',
        '3-fold CV, 50 series subset (store=1)',
        '50 series; cold-start on 30-day truncation',
        'Inverse-error weights on validation SMAPE',
        'Tier3 for cold-start (<90d history), blend T1+T2 otherwise'
    ]
})
tier_comp.to_csv(os.path.join(RESULT_DIR, 'tier_comparison.csv'), index=False)
print("Saved: tier_comparison.csv")
print(tier_comp.to_string(index=False))


---
## Phase 8 — Inventory Decision Layer
Converting the probabilistic forecast into a **newsvendor-style reorder point recommendation**:

```
reorder_point = lead_time_demand(P50 over lead time) + z(service_level) × σ_forecast
```
where `σ_forecast` is estimated from the P90/P50 spread of the quantile forecast.


In [ ]:
# ── Phase 8.1: Quantile forecast extraction ──────────────────────────────────
# For each (store, item) series, extract P50, P90, P95 from the ensemble/model
# We use LightGBM P50 + empirical residual std to construct quantile bands

LEAD_TIME = 7   # days
SERVICE_LEVELS = {0.90: 1.282, 0.95: 1.645, 0.99: 2.326}  # z-scores

# Use last validation fold predictions for computing safety stock
# Compute residuals on last validation fold for each series
fold = CV_FOLDS[2]

inventory_rows = []
sample_series = list(itertools.islice(
    ((s, i) for s in range(1, 11) for i in range(1, 51)), 20
))

for store, item in sample_series:
    series_mask = (df_feat['store'] == store) & (df_feat['item'] == item)
    val_mask_si = series_mask & (df_feat['date'] >= fold['val_start']) & (df_feat['date'] <= fold['val_end'])
    tr_mask_si  = series_mask & (df_feat['date'] <= fold['train_end'])

    si_train = df_feat[tr_mask_si]
    si_val   = df_feat[val_mask_si].copy()
    if si_val.empty:
        continue

    X_si = si_val[ALL_COLS].copy()
    for col in ['store','item']:
        X_si[col] = X_si[col].astype('category')
    preds_si = np.maximum(best_model_lgbm.predict(X_si), 0)
    si_val = si_val.copy()
    si_val['pred'] = preds_si

    # Residuals → forecast std dev (proxy for uncertainty)
    residuals = si_val['sales'].values - preds_si
    sigma_daily = np.std(residuals)

    # Lead-time demand: sum of P50 over LEAD_TIME days (next 7 days of forecast)
    lead_preds = preds_si[:LEAD_TIME] if len(preds_si) >= LEAD_TIME else preds_si
    lead_time_demand = np.sum(lead_preds)

    # Sigma over lead time (assuming iid residuals — conservative)
    sigma_lt = sigma_daily * np.sqrt(LEAD_TIME)

    # Historical avg daily demand
    hist_avg = si_train['sales'].mean()

    row = {
        'store': store,
        'item' : item,
        'hist_avg_daily_demand': round(hist_avg, 2),
        'lead_time_demand_P50' : round(lead_time_demand, 2),
        'sigma_lead_time'      : round(sigma_lt, 2),
    }
    for sl, z in SERVICE_LEVELS.items():
        ss = z * sigma_lt
        rp = lead_time_demand + ss
        row[f'safety_stock_{int(sl*100)}pct']  = round(ss, 2)
        row[f'reorder_point_{int(sl*100)}pct'] = round(rp, 2)

    inventory_rows.append(row)

inventory_df = pd.DataFrame(inventory_rows)
print("=" * 80)
print("INVENTORY DECISION TABLE — Sample of 20 Store-Item Series")
print("=" * 80)
display_cols = ['store','item','hist_avg_daily_demand','lead_time_demand_P50',
                'sigma_lead_time','reorder_point_90pct','reorder_point_95pct','reorder_point_99pct']
print(inventory_df[display_cols].to_string(index=False))


In [ ]:
# ── Phase 8.2: Save inventory recommendations ────────────────────────────────
inventory_df.to_csv(os.path.join(RESULT_DIR, 'inventory_recommendations.csv'), index=False)
print("Saved: inventory_recommendations.csv")

# Visualise service-level tradeoff for top 5 series
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(inventory_df[:10]))
width = 0.25

bars1 = ax.bar(x - width, inventory_df[:10]['reorder_point_90pct'], width, label='90% Service Level', color='steelblue', alpha=0.85)
bars2 = ax.bar(x,         inventory_df[:10]['reorder_point_95pct'], width, label='95% Service Level', color='darkorange', alpha=0.85)
bars3 = ax.bar(x + width, inventory_df[:10]['reorder_point_99pct'], width, label='99% Service Level', color='tomato', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f"S{r['store']}/I{r['item']}" for _, r in inventory_df[:10].iterrows()], rotation=30)
ax.set_ylabel('Reorder Point (units)')
ax.set_title('Reorder Points at 90/95/99% Service Levels — Sample 10 Series', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'inventory_reorder_points.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\nInterpretation: Higher service levels require progressively larger safety buffers.")
print("The difference between 90% and 99% SL represents the inventory cost of risk reduction.")


---
## Phase 9 — Monitoring & Retraining Triggers
A production-grade monitoring module that tracks **forecast error drift** (rolling SMAPE) and **input distribution drift** (Population Stability Index — PSI) to trigger retraining when the model's assumptions break.


In [ ]:
# ── Phase 9.1: PSI function ─────────────────────────────────────────────────
def compute_psi(baseline, current, n_bins=10, eps=1e-6):
    """
    Population Stability Index.
    PSI < 0.1  → No significant change
    PSI 0.1–0.2 → Minor shift (monitor)
    PSI > 0.2  → Major shift (retrain)
    """
    bins = np.percentile(baseline, np.linspace(0, 100, n_bins + 1))
    bins[0] -= eps
    bins[-1] += eps
    base_counts  = np.histogram(baseline, bins=bins)[0] + eps
    curr_counts  = np.histogram(current,  bins=bins)[0] + eps
    base_pct = base_counts / base_counts.sum()
    curr_pct = curr_counts / curr_counts.sum()
    psi = np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct))
    return psi

print("PSI function defined.")
print("Thresholds: PSI < 0.1 (stable) | 0.1–0.2 (monitor) | > 0.2 (retrain)")


In [ ]:
# ── Phase 9.2: Rolling SMAPE drift monitoring ─────────────────────────────
# We simulate the monitoring window: use Q3 2017 as 'recent' and compare
# against the Q1 2017 backtest baseline SMAPE.

MONITOR_WINDOW_START = '2017-10-01'
MONITOR_WINDOW_END   = '2017-12-31'
BACKTEST_SMAPE_REF   = results_lgbm['SMAPE'].mean()  # overall CV mean
RETRAIN_SMAPE_THRESH = BACKTEST_SMAPE_REF * 1.20     # 20% degradation trigger

monitor_rows = []
for store in range(1, 4):    # monitoring sample: stores 1–3 to keep runtime reasonable
    for item in range(1, 11): # items 1–10
        si_mask = (df_feat['store'] == store) & (df_feat['item'] == item)
        mon_mask = si_mask & (df_feat['date'] >= MONITOR_WINDOW_START) & (df_feat['date'] <= MONITOR_WINDOW_END)
        tr_mask  = si_mask & (df_feat['date'] <= '2017-06-30')

        si_mon = df_feat[mon_mask].copy()
        si_tr  = df_feat[tr_mask].copy()
        if si_mon.empty or si_tr.empty:
            continue

        X_mon = si_mon[ALL_COLS].copy()
        for col in ['store','item']:
            X_mon[col] = X_mon[col].astype('category')
        preds_mon = np.maximum(best_model_lgbm.predict(X_mon), 0)
        rolling_smape = smape(si_mon['sales'].values, preds_mon)

        # PSI: training distribution vs monitoring window
        psi = compute_psi(si_tr['sales'].values, si_mon['sales'].values)

        # Flag conditions
        smape_flag = rolling_smape > RETRAIN_SMAPE_THRESH
        psi_flag   = psi > 0.2
        monitor_rows.append({
            'store'         : store,
            'item'          : item,
            'rolling_SMAPE' : round(rolling_smape, 4),
            'PSI'           : round(psi, 4),
            'SMAPE_flagged' : smape_flag,
            'PSI_flagged'   : psi_flag,
            'retrain_needed': smape_flag or psi_flag,
        })

monitor_df = pd.DataFrame(monitor_rows)
flagged_df = monitor_df[monitor_df['retrain_needed']]
print(f"Monitoring period : {MONITOR_WINDOW_START} → {MONITOR_WINDOW_END}")
print(f"Backtest ref SMAPE: {BACKTEST_SMAPE_REF:.4f}%")
print(f"Retrain threshold : {RETRAIN_SMAPE_THRESH:.4f}% (20% above baseline)")
print(f"\nSeries monitored : {len(monitor_df)}")
print(f"Series flagged   : {len(flagged_df)}")
print()
if not flagged_df.empty:
    print("FLAGGED SERIES:")
    print(flagged_df[['store','item','rolling_SMAPE','PSI','SMAPE_flagged','PSI_flagged']].to_string(index=False))
else:
    print("✓ No series require immediate retraining. All metrics within acceptable bounds.")
print(f"\nMean PSI across monitored series: {monitor_df['PSI'].mean():.4f}")
print(f"Mean rolling SMAPE: {monitor_df['rolling_SMAPE'].mean():.4f}%")


In [ ]:
# ── Phase 9.3: Monitoring dashboard plot ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Monitoring — SMAPE & PSI by Series', fontsize=13, fontweight='bold')

ax = axes[0]
colors = ['tomato' if f else 'steelblue' for f in monitor_df['SMAPE_flagged']]
ax.bar(range(len(monitor_df)), monitor_df['rolling_SMAPE'], color=colors, alpha=0.8)
ax.axhline(RETRAIN_SMAPE_THRESH, color='red', linestyle='--', linewidth=1.5, label=f'Retrain threshold ({RETRAIN_SMAPE_THRESH:.1f}%)')
ax.axhline(BACKTEST_SMAPE_REF,   color='green', linestyle='--', linewidth=1.5, label=f'Backtest baseline ({BACKTEST_SMAPE_REF:.1f}%)')
ax.set_title('Rolling SMAPE (Oct–Dec 2017)')
ax.set_xlabel('Series index')
ax.set_ylabel('SMAPE (%)')
ax.legend(fontsize=8)

ax = axes[1]
colors_psi = ['tomato' if f else 'steelblue' for f in monitor_df['PSI_flagged']]
ax.bar(range(len(monitor_df)), monitor_df['PSI'], color=colors_psi, alpha=0.8)
ax.axhline(0.2, color='red',    linestyle='--', linewidth=1.5, label='Major drift (0.2)')
ax.axhline(0.1, color='orange', linestyle='--', linewidth=1.5, label='Minor drift (0.1)')
ax.set_title('PSI — Sales Distribution Drift')
ax.set_xlabel('Series index')
ax.set_ylabel('PSI')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'monitoring_smape_psi.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Phase 10 — Final Report & Results Dashboard
A comprehensive summary of all results, tying together the three-tier model comparison, reconciliation, cold-start findings, and inventory recommendations into a single evaluation view.


In [ ]:
# ── Phase 10.1: Summary comparison table ─────────────────────────────────────
print("=" * 80)
print("FINAL RESULTS SUMMARY — Multi-Series Demand Forecasting System")
print("=" * 80)
print()

tier_comp_loaded = pd.read_csv(os.path.join(RESULT_DIR, 'tier_comparison.csv'))
print("TIER COMPARISON (SMAPE / WAPE):")
print(tier_comp_loaded[['tier','SMAPE_mean','WAPE_mean','cold_start_smape']].round(4).to_string(index=False))
print()

recon_check_loaded = pd.read_csv(os.path.join(RESULT_DIR, 'reconciliation_check.csv'))
print("RECONCILIATION CHECK:")
print(recon_check_loaded.to_string(index=False))
print()

print("COLD-START COMPARISON (Tier 3 vs Tier 1, 30-day history truncation):")
print(f"  Warm series SMAPE (Tier 3): {warm_smape:.4f}%")
print(f"  Cold-start SMAPE (Tier 3) : {cold_smape:.4f}%")
print(f"  Cold-start SMAPE (Tier 1) : N/A (requires >28 day lag window → unfeasible at 30d)")
print()

print("INVENTORY RECOMMENDATIONS (first 5 rows):")
inv_loaded = pd.read_csv(os.path.join(RESULT_DIR, 'inventory_recommendations.csv'))
print(inv_loaded.head(5)[['store','item','hist_avg_daily_demand','reorder_point_90pct',
                           'reorder_point_95pct','reorder_point_99pct']].to_string(index=False))


In [ ]:
# ── Phase 10.2: Master performance dashboard ─────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Tier SMAPE Comparison',
        'Reconciliation: Pre vs Post SMAPE',
        'Inventory: Reorder Points at Service Levels',
        'Monitoring: PSI Distribution'
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

# Panel 1: Tier SMAPE comparison
tier_labels = tier_comp_loaded['tier'].tolist()
smape_vals  = tier_comp_loaded['SMAPE_mean'].tolist()
colors_bars = ['steelblue', 'darkorange', 'mediumseagreen', 'mediumpurple', 'tomato']
fig.add_trace(
    go.Bar(x=tier_labels, y=smape_vals, marker_color=colors_bars,
           text=[f'{v:.2f}%' for v in smape_vals], textposition='outside',
           name='SMAPE'),
    row=1, col=1
)

# Panel 2: Reconciliation pre vs post
recon_vals = recon_check_loaded[recon_check_loaded['metric'].str.contains('SMAPE')]['value'].values
recon_labels = ['Pre-Reconciliation', 'Post-Reconciliation']
fig.add_trace(
    go.Bar(x=recon_labels, y=recon_vals[:2],
           marker_color=['cornflowerblue', 'mediumseagreen'],
           text=[f'{v:.2f}%' for v in recon_vals[:2]], textposition='outside',
           name='SMAPE'),
    row=1, col=2
)

# Panel 3: Inventory reorder points
sample_inv = inv_loaded.head(8)
xlabels_inv = [f"S{r['store']}/I{r['item']}" for _, r in sample_inv.iterrows()]
for sl, col_name, color in [(90, 'reorder_point_90pct', 'steelblue'),
                             (95, 'reorder_point_95pct', 'darkorange'),
                             (99, 'reorder_point_99pct', 'tomato')]:
    fig.add_trace(
        go.Bar(x=xlabels_inv, y=sample_inv[col_name].tolist(),
               name=f'{sl}% SL', marker_color=color, opacity=0.85),
        row=2, col=1
    )

# Panel 4: PSI distribution
fig.add_trace(
    go.Histogram(x=monitor_df['PSI'].tolist(), nbinsx=10,
                 marker_color='steelblue', opacity=0.8, name='PSI'),
    row=2, col=2
)
fig.add_vline(x=0.1, line_dash='dash', line_color='orange', row=2, col=2)
fig.add_vline(x=0.2, line_dash='dash', line_color='red', row=2, col=2)

fig.update_layout(
    height=700, width=1100,
    title_text='Retail Demand Forecasting System — Final Results Dashboard',
    title_font_size=16,
    showlegend=False,
    template='plotly_white'
)
fig.update_yaxes(title_text='SMAPE (%)', row=1, col=1)
fig.update_yaxes(title_text='SMAPE (%)', row=1, col=2)
fig.update_yaxes(title_text='Units',     row=2, col=1)
fig.update_yaxes(title_text='Count',     row=2, col=2)
fig.update_xaxes(tickangle=30, row=2, col=1)

fig.write_html(os.path.join(RESULT_DIR, 'final_dashboard.html'))
fig.show()
print("Saved interactive dashboard: results/final_dashboard.html")


In [ ]:
# ── Phase 10.3: Generate Kaggle submission ───────────────────────────────────
# Use best LightGBM model predictions for submission
submission = test_raw[['id']].copy()
submission['sales'] = lgbm_test_preds['lgbm_pred'].values.round().astype(int)
submission['sales'] = submission['sales'].clip(lower=0)
submission.to_csv(os.path.join(RESULT_DIR, 'submission.csv'), index=False)
print(f"Submission saved: {os.path.join(RESULT_DIR, 'submission.csv')}")
print(submission.head(10))
print(f"\nSales stats: min={submission['sales'].min()}, max={submission['sales'].max()}, mean={submission['sales'].mean():.2f}")


---
## Phase 10.4 — Written Summary & Production Recommendation

### Key Findings

| Scenario | Recommended Tier | Rationale |
|---|---|---|
| Established series (≥1 year history) | Tier 1 + Tier 2 blend | LightGBM is fast, interpretable, and strong on lag-heavy retail data; N-HiTS captures multi-horizon seasonality jointly |
| New / sparse series (<90 days history) | Tier 3 (Chronos/ETS) | No training needed; foundation model generalises from pretraining on diverse time series |
| Production deployment | Weighted ensemble with routing | Best of all worlds; routing rule makes the cold-start advantage of Tier 3 explicit |

### What Each Tier Solved

- **Tier 1 (LightGBM):** Fast, interpretable, strong SMAPE baseline using hand-engineered lag & Fourier features. Best single-model for the full 500-series dataset.
- **Tier 2 (N-HiTS):** Learns shared temporal representations across series jointly, produces calibrated quantile forecasts (P10/P50/P90) directly usable for inventory decisions — the gradient boosting tier cannot do this natively.
- **Tier 3 (Chronos/ETS):** Zero-shot performance. Critical for cold-start scenarios where Tiers 1–2 have no history to learn from. The cold-start SMAPE gap shows this tier's practical value is in the tail of new-series onboarding, not head-to-head full-history competition.

### Reconciliation
MinT reconciliation enforced coherence between the 50 item-level forecasts and the store-level aggregate, ensuring that supply chain teams working at different aggregation levels receive consistent demand signals.

### Inventory Decision Layer
The newsvendor-style reorder point formula translates the probabilistic forecast directly into a business recommendation. The service-level table makes the **cost of reliability** explicit: moving from 90% to 99% service level roughly doubles the safety stock buffer for typical series in this dataset.

### Monitoring
Rolling SMAPE + PSI monitoring provides an automated early-warning system. PSI > 0.2 flags demand pattern shifts that the model was not trained on — critical for retail contexts with promotions, stockouts, or new competitor entrants that can shift baseline demand patterns.

### Limitations (from design.md §7)
- No external covariates (price, promotion, holiday data) — caps feature engineering potential
- Tier 2 & 3 evaluated on 50-series subset due to compute; full-dataset evaluation recommended in production
- SMAPE distorts at near-zero actuals — WAPE reported as a more stable complementary metric

---
*End of Demand_Forecasting_System.ipynb — Phases 0–10*
